# 04.6 — The multi-head classifier

The decoding task isn't one classification — it's **several at once**. Each quaddle object has multiple feature dimensions (the production `Dimension` target decodes 4), and the model predicts every dimension in parallel from the same latent representation. The structure that does this is the **multi-head classifier**: one small output head per target dimension, collected in an `nn.ModuleList`.

This notebook covers:

* Why multi-output (multi-task) rather than one classifier.
* `nn.ModuleList` of heads — and the registration reason it must be a ModuleList.
* The list-of-logits output and how the loss consumes it.
* `MultiHeadClassifier` (Logistic) vs `DeepLSTMClassifier` (the Optimal one).

**Prerequisites:** [02.6 nn.Module](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb) (the ModuleList trap), [04.5 the bottleneck](04.5_the_bottleneck.ipynb).

## Section 1 — What MATLAB does

The MATLAB classifier branches into one output path per target dimension, each ending in its own softmax over that dimension's class count:

```matlab
for d = 1:NumDimensions
    branch = [fullyConnectedLayer(NumClasses(d)); softmaxLayer];
    lgraph = addLayers(lgraph, branch);
    lgraph = connectLayers(lgraph, 'sharedLatent', ['fc_dim' num2str(d)]);
end
```

A shared latent feeds `NumDimensions` independent heads; each head predicts its own dimension's class. The loss sums each head's cross-entropy. The Python port is the same shape with `nn.ModuleList` standing in for the `addLayers`/`connectLayers` wiring.

## Section 2 — The Python concepts you need

### 2.1 — One head per dimension

In [ ]:
import torch
from neural_data_decoding.models.classifier import MultiHeadClassifier

# 3 output dimensions with 3, 4, and 2 classes respectively:
clf = MultiHeadClassifier(in_features=8, num_classes_per_dim=[3, 4, 2])
x = torch.randn(5, 8)               # (batch=5, latent_features=8)
logits = clf(x)

print("output type:", type(logits).__name__, "of", len(logits), "heads")
for d, head_logits in enumerate(logits):
    print(f"  dimension {d}: {tuple(head_logits.shape)}  ({head_logits.shape[1]} classes)")

The output is a **list** — one logits tensor per dimension, each `(batch, num_classes_for_that_dim)`. Not stacked into one tensor, because the dimensions have *different* class counts (3, 4, 2) — a ragged set that no single tensor holds cleanly. The list keeps them separate, and the multi-head loss iterates it ([04.7](04.7_weighted_classification_loss.ipynb)).

### 2.2 — Why `nn.ModuleList` (the trap, revisited)

The heads live in an `nn.ModuleList`. This is the exact lesson from [02.6 §2.2](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb), and here's why it's load-bearing: with a plain Python list, the heads' parameters would be **invisible to `parameters()`**, so the optimizer would never train them — silently:

In [ ]:
import torch.nn as nn

class BrokenMultiHead(nn.Module):
    def __init__(self, in_f, classes):
        super().__init__()
        self.heads = [nn.Linear(in_f, c) for c in classes]     # plain list — WRONG

class CorrectMultiHead(nn.Module):
    def __init__(self, in_f, classes):
        super().__init__()
        self.heads = nn.ModuleList(nn.Linear(in_f, c) for c in classes)

print("Broken  (plain list) params:", sum(p.numel() for p in BrokenMultiHead(8, [3,4,2]).parameters()))
print("Correct (ModuleList) params:", sum(p.numel() for p in CorrectMultiHead(8, [3,4,2]).parameters()))
print("The production MultiHeadClassifier uses ModuleList:",
      type(clf.heads).__name__)

Zero parameters for the broken version — every head silently untrained. For a *multi*-head model this trap is especially cruel: the shared trunk trains, loss decreases, and you'd swear it works — but the heads that actually make the predictions never move. `nn.ModuleList` is the fix, and it's why every collection-of-submodules in this codebase uses it.

### 2.3 — Building heads with a comprehension

In [ ]:
# Each head is Linear(latent → that dimension's class count) — a comprehension over the class list:
in_features, num_classes_per_dim = 8, [3, 4, 2]
heads = nn.ModuleList(nn.Linear(in_features, c) for c in num_classes_per_dim)
for d, head in enumerate(heads):
    print(f"  head {d}: Linear({in_features} → {head.out_features})")

Module construction is just Python ([01.2 §2.6](../01_python_for_matlab_users/01.2_control_flow.ipynb)'s comprehension, [02.6](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb)'s modules-are-values) — the `nn.ModuleList` wrapper is the only thing distinguishing this from a throwaway list.

### 2.4 — The two classifiers

`MultiHeadClassifier` (used by the `Logistic` model) reads the latent directly — one Linear head per dimension, no recurrence. The **Optimal** model uses `DeepLSTMClassifier`, which puts an LSTM stack ([04.3](04.3_rnn_building_blocks.ipynb)) *before* the per-dimension heads, so each prediction sees the full window sequence. Both end in the same multi-head structure and return the same list-of-logits — swappable via `cfg.classifier_name`:

In [ ]:
import neural_data_decoding.models
from neural_data_decoding.models.registry import list_classifiers, build_classifier
print("registered classifiers:", list_classifiers())

logistic = build_classifier("Logistic", {"in_features": 8, "num_classes_per_dim": [3, 4, 2]})
out = logistic(torch.randn(5, 8))
print("Logistic classifier → list of", len(out), "logits tensors:", [tuple(o.shape) for o in out])

## Section 3 — The `neural_data_decoding` implementation

`MultiHeadClassifier` — the `ModuleList` of heads and the list-returning forward:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/classifier.py").read_text().splitlines()
cls = next(j for j, line in enumerate(src) if line.startswith("class MultiHeadClassifier"))
# find the ModuleList assignment inside __init__ and print a window around it
ml = next(j for j in range(cls, len(src)) if "ModuleList" in src[j])
print(f"{cls + 1:4} | {src[cls]}")
print("     |   ...")
for k in range(ml - 3, min(ml + 6, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `self.heads = nn.ModuleList(...)` built from a comprehension over `num_classes_per_dim` — §2.2/§2.3 in production.
* Each head is a `Linear(in_features → num_classes[d])` with He init ([04.8](04.8_weight_initialization_he_vs_pytorch_defaults.ipynb)) — the same explicit-init discipline as the bottleneck.
* `forward` returns `[head(x) for head in self.heads]` — a list, deliberately not a stacked tensor (§2.1). The downstream multi-head cross-entropy ([04.7](04.7_weighted_classification_loss.ipynb)) expects exactly this list.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the total head parameter count

For `MultiHeadClassifier(in_features=16, num_classes_per_dim=[2, 5, 3, 4])`, compute the total head parameters by hand (each head: `in×classes + classes` biases) and verify.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
clf = MultiHeadClassifier(in_features=16, num_classes_per_dim=[2, 5, 3, 4])
by_hand = sum(16 * c + c for c in [2, 5, 3, 4])
print(f"by hand: {by_hand}   actual: {sum(p.numel() for p in clf.parameters())}")

### Exercise 4.2 — consume the output

Given the list-of-logits from a 3-dimension classifier, compute the predicted class for each dimension (argmax) for the first sample in the batch — the shape of what the CM-table writer records.

In [ ]:
# Reveal:
logits = MultiHeadClassifier(8, [3, 4, 2])(torch.randn(5, 8))
preds = [head_logits.argmax(dim=1)[0].item() for head_logits in logits]
print("predicted class per dimension (sample 0):", preds)

## Section 5 — Common errors

### Some output dimensions never improve during training

The plain-list trap (§2.2) — heads in a Python list are invisible to the optimizer. Confirm `type(model.heads).__name__ == "ModuleList"` and that `sum(p.numel() for p in model.parameters())` includes the heads.

### `RuntimeError` stacking the output

You tried `torch.stack(logits)` on heads with different class counts (ragged). Keep them as a list; the loss iterates it.

### The loss function gets the wrong shape

The multi-head loss expects a *list* of per-dimension logits, not a single tensor. If you flattened or concatenated the heads, unflatten them back to the list.

### `IndexError` matching logits to targets

The number of heads must equal the number of target dimensions, and each head's class count must cover its dimension's labels. A mismatch between `num_classes_per_dim` and the actual data surfaces here — the dataset's `num_classes_per_dim` is the source of truth ([03.1](../03_data_pipeline/03.1_dataset_vs_filedatastore.ipynb)).

### Switching Logistic → Deep LSTM changes the input shape needed

`MultiHeadClassifier` takes `(batch, features)`; `DeepLSTMClassifier` takes a `(batch, sequence, features)` — it wants the window sequence the bottleneck preserved ([04.5 §2.2](04.5_the_bottleneck.ipynb)). Feed the right rank for the classifier you chose.

## Section 6 — Further reading

- [Multi-task learning overview](https://ruder.io/multi-task/) — why shared representations + task-specific heads.
- [nn.ModuleList docs](https://pytorch.org/docs/stable/generated/torch.nn.ModuleList.html) — the registration guarantee.
- [`src/neural_data_decoding/models/classifier.py`](../../src/neural_data_decoding/models/classifier.py) — both classifiers, heads first.

Next up: **[04.7 weighted classification loss](04.7_weighted_classification_loss.ipynb)** — how `WeightedLoss='Inverse'` counters class imbalance across those heads.